In [ ]:
import sys
print(sys.executable)

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

print("LangChain imported successfully!")

In [ ]:
import os

print(os.listdir("pdfs_visa"))

In [ ]:
import os

print(os.getcwd())

In [ ]:
import os

print(os.listdir())

## Data Loading - Data Ingestion

In [ ]:
import os

for country in os.listdir("pdfs_visa"):
    country_path = os.path.join("pdfs_visa", country)

    print(f"\n\U0001F4C1 {country}")

    if os.path.isdir(country_path):
        print(os.listdir(country_path))

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("pdfs_visa")
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")

In [ ]:
print(documents[0].metadata)
print("=" * 80)
print(documents[0].page_content[:1000])

## Text Processing - Recursive Character Text Splitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

In [ ]:
print(chunks[0].page_content)

## HuggingFaceEmbedding Model Loaded

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

## Create Vector DB and Embedding

In [ ]:
from langchain_chroma import Chroma

In [ ]:
import os

for chunk in chunks:
    source = chunk.metadata["source"]

    # Country = folder name
    country = os.path.basename(os.path.dirname(source))

    # PDF name
    document = os.path.basename(source)

    chunk.metadata["country"] = country
    chunk.metadata["document"] = document

In [ ]:
print(chunks[0].metadata)

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="visas_db"
)

In [ ]:
print(vector_db._collection.count())

In [ ]:
print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

In [ ]:
for i in range(5):
    print(f"Chunk {i+1} length:", len(chunks[i].page_content))
    print(chunks[i].metadata)
    print("-" * 80)

## Retriever

In [ ]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

In [ ]:
query = "What are the financial requirements for Australia?"

In [ ]:
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results, 1):
    print("=" * 80)
    print(f"Result {i}")
    print("Country :", doc.metadata["country"])
    print("Document:", doc.metadata["document"])
    print("Page    :", doc.metadata["page"] + 1)
    print("-" * 80)
    print(doc.page_content)
    print()

## 2nd Half of the Architecture

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

In [ ]:
!pip install langchain-openai langchain-groq

In [ ]:
import os

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

In [ ]:
import os

# NOTE: Do not hardcode API keys in notebooks you share or commit to version control.
# Set these as environment variables in your shell, or use a .env file with python-dotenv instead.
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

## LLM Provider Helper Function

A helper function is created to abstract the LLM initialization logic. This allows the application to switch between different LLM providers (Groq and OpenAI) without modifying the rest of the RAG pipeline.

- **Groq** is used during development for faster inference and lower cost.
- **OpenAI** can be used for the final demo or deployment to leverage higher-quality responses.

This approach improves code reusability, maintainability, and makes it easy to experiment with multiple LLM providers by changing only a single parameter.

In [ ]:
def get_llm(provider="groq"):
    """
    Returns the selected LLM.

    provider:
        "groq"
        "openai"
    """
    provider = provider.lower()

    if provider == "groq":
        return ChatGroq(
            model="llama-3.3-70b-versatile",
            temperature=0
        )

    elif provider == "openai":
        return ChatOpenAI(
            model="gpt-4.1-mini",
            temperature=0
        )

    else:
        raise ValueError("Provider must be either 'groq' or 'openai'")

In [ ]:
llm = get_llm("groq")

## Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI Visa Assistant.

Your task is to answer the user's question ONLY using the provided context.

Rules:
1. Use only the provided context.
2. Do not make up any information.
3. If the answer is not available in the context, respond with:
   "I couldn't find this information in the provided documents."
4. Answer in a clear and professional manner.

Context:
{context}

Question:
{question}

Answer:
""")

## Retriever Context

In [ ]:
query = "What are the financial requirements for Australia?"
results = retriever.invoke(query)

In [ ]:
context = "\n\n".join(doc.page_content for doc in results)

In [ ]:
print(context)

In [ ]:
messages = prompt.invoke({
    "context": context,
    "question": query
})

In [ ]:
print(messages)

In [ ]:
response = llm.invoke(messages)
print(response.content)